# 教程: 参数敏感性分析

## 目的
模型中通常包含一些不确定或需要率定的参数。参数敏感性分析（Sensitivity Analysis）是了解模型行为、确定关键参数的重要步骤。其目的是系统性地改变一个或多个参数的值，并观察模型输出（如洪峰流量、总径流量）如何响应这些变化。

此笔记本将：
1.  选择一个简单的水文模型作为分析对象。
2.  演示一种简单的一维敏感性分析方法（“一次改变一个” - OAT），即每次只改变一个参数，而保持其他参数不变。
3.  运行多次模拟，收集不同参数值对应的模型输出。
4.  通过绘图来直观地展示模型的输出对每个参数的敏感程度。

## 1. 环境设置和模型准备

我们选择一个由`SCSCurveNumberModule`和`SimpleRouting`组成的简单水文模型作为我们的分析对象。这个模型有三个关键参数：`CN`, `k_q`, `k_s`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

# --- 定义一个辅助函数来运行模型 ---
def run_hydro_model(params, rainfall, pet):
    runoff_module = SCSCurveNumberModule(CN=params['CN'])
    routing_module = SimpleRouting(k_q=params['k_q'], k_s=params['k_s'])
    model = HydrologicalModel(runoff_module, routing_module)
    
    outflows = []
    for i in range(len(rainfall)):
        outflow = model.run(rainfall[i], pet[i])
        outflows.append(outflow)
    return np.array(outflows)

# --- 定义基准参数和输入 ---
base_params = {'CN': 75, 'k_q': 0.8, 'k_s': 0.2}
sample_rainfall = np.array([0, 5, 10, 30, 50, 20, 10, 5, 0, 0, 15, 35, 10, 0])
sample_pet = np.full_like(sample_rainfall, 1.0)

## 2. 执行敏感性分析

我们对每个参数定义一个取值范围，然后循环运行模拟。在每个循环中，我们只改变一个参数，并记录下该参数值对应的洪峰流量。

In [ ]:
sensitivity_results = {
    'CN': {'values': [], 'peak_flows': []},
    'k_q': {'values': [], 'peak_flows': []},
    'k_s': {'values': [], 'peak_flows': []}
}

# --- 1. 分析 CN 的敏感性 ---
print("分析参数: CN")
cn_values = np.linspace(50, 95, 10)
for cn in cn_values:
    params = base_params.copy()
    params['CN'] = cn
    outflow = run_hydro_model(params, sample_rainfall, sample_pet)
    sensitivity_results['CN']['values'].append(cn)
    sensitivity_results['CN']['peak_flows'].append(np.max(outflow))

# --- 2. 分析 k_q 的敏感性 ---
print("分析参数: k_q")
kq_values = np.linspace(0.1, 1.0, 10)
for kq in kq_values:
    params = base_params.copy()
    params['k_q'] = kq
    outflow = run_hydro_model(params, sample_rainfall, sample_pet)
    sensitivity_results['k_q']['values'].append(kq)
    sensitivity_results['k_q']['peak_flows'].append(np.max(outflow))

# --- 3. 分析 k_s 的敏感性 ---
print("分析参数: k_s")
ks_values = np.linspace(0.01, 0.5, 10)
for ks in ks_values:
    params = base_params.copy()
    params['k_s'] = ks
    outflow = run_hydro_model(params, sample_rainfall, sample_pet)
    sensitivity_results['k_s']['values'].append(ks)
    sensitivity_results['k_s']['peak_flows'].append(np.max(outflow))

print("\n敏感性分析运行完毕。")

## 3. 可视化结果

现在我们为每个参数绘制一张图，横坐标是参数的值，纵坐标是洪峰流量。图线的斜率直接反映了模型对该参数的敏感程度。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Parameter Sensitivity Analysis (Response: Peak Flow)', fontsize=16)

# CN plot
ax1 = axes[0]
ax1.plot(sensitivity_results['CN']['values'], sensitivity_results['CN']['peak_flows'], 'r-o')
ax1.set_title('Sensitivity to CN')
ax1.set_xlabel('Curve Number (CN)')
ax1.set_ylabel('Peak Flow (mm)')
ax1.grid(True)

# k_q plot
ax2 = axes[1]
ax2.plot(sensitivity_results['k_q']['values'], sensitivity_results['k_q']['peak_flows'], 'g-s')
ax2.set_title('Sensitivity to k_q (Quick Flow Coeff.)')
ax2.set_xlabel('k_q')
ax2.grid(True)

# k_s plot
ax3 = axes[2]
ax3.plot(sensitivity_results['k_s']['values'], sensitivity_results['k_s']['peak_flows'], 'b-^')
ax3.set_title('Sensitivity to k_s (Slow Flow Coeff.)')
ax3.set_xlabel('k_s')
ax3.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 结果解读

从上面的三张图中，我们可以得出结论：
- **对`CN`最敏感**: `CN`图的斜率最大，表明`CN`值的变化对洪峰流量的影响最大。这符合预期，因为`CN`直接控制了总产流量的多少。
- **对`k_q`次之**: `k_q`（快速流出流系数）的改变也对洪峰有显著影响。`k_q`越大，快速流越多，洪峰也越高。
- **对`k_s`最不敏感**: `k_s`（慢速流出流系数）图的斜率非常平缓，几乎为零。这说明在这个模型和输入条件下，洪峰流量对慢速流（基流）部分的参数变化不敏感。

**结论**: 在对这个模型进行参数率定时，我们应该优先关注和校准`CN`和`k_q`这两个参数，而`k_s`则可以被视为一个次要参数或固定为一个经验值。